Descripción del notebook

In [ ]:
# coding=utf-8
# Título: Práctica 1-Búsqueda de imagenes de satélite Sentinel 2A con Copernicus Data Space Ecosystem API
# Requerimientos: Python 3x
# Librerías: geopangas, pandas, request, json, datetime, time
# Autor: Mauricio Tabares Mosquera
# Fecha: 2026-04-07

Importar librerías

In [ ]:
import requests, json, datetime, geopandas as gpd
from shapely.geometry import shape
from time import strftime
print("Librerías cargadas")

Parámetros de usuario

In [ ]:
# Variables de usuario
fecha_actual = datetime.datetime.today()
delta_30_dias = datetime.timedelta(days=30)
fecha_anterior = fecha_actual - delta_30_dias
area_interes = [0.159413, 40.522982, 3.322251, 42.861523] # Extensión de Cataluña

# Formatear la fecha
fecha_actual = fecha_actual.strftime('%Y-%m-%d') # Con formato Año-Mes-Día
fecha_anterior = fecha_anterior.strftime('%Y-%m-%d')
print(f"Fecha actual: {fecha_actual} | Fecha anterior: {fecha_anterior}")

# Diccionario con parámetros
params = {
    "collection": "sentinel-2-l2a",
    "fecha_inicio": fecha_anterior,
    "fecha_final": fecha_actual,
    "cobertura_nubes_min": 0,
    "cobertura_nubes_max": 5,
    "bbox": area_interes,
    "limit": 100
}

print("Variables de usuario establecidas")

Recuperar imágenes Sentinel-2A

In [ ]:
# Inicio de script
print("Consulta de imágenes iniciada: ".upper() + strftime("%Y-%m-%d %H:%M:%S"))

# Parámetros en formato de diccionario
get_params = {
    "collections": params["collection"],
    "bbox": ",".join(map(str, params["bbox"])),
    "datetime": f"{params['fecha_inicio']}T00:00:00Z/{params['fecha_final']}T23:59:59Z",
    "query": json.dumps({
        "eo:cloud_cover": {
            "gte": params["cobertura_nubes_min"],
            "lte": params["cobertura_nubes_max"]
        }
    }),
    "limit": params["limit"]
}

# Crear endpoint del servicio
url = "https://stac.dataspace.copernicus.eu/v1/search"
# Enviar petición con parámetros
response = requests.get(url, params=get_params)
print("Parámetros de consulta establecidos")

# Crear un Geodataframe con el resultado de la consulta
if response.status_code == 200:
    print("Imágenes Sentinel-2A recuperadas")
    items = response.json().get("features", [])
    
    # Crear lista de diccionarios 'properties' y 'geometry'
    data_list = []
    for item in items:
        # Extraemos propiedades y añadimos ID y geometría
        row = item['properties'].copy()
        row['id'] = item['id']
        # Convertir el resultado a un objeto shape de Shapely
        row['geometry'] = shape(item['geometry']) 
        data_list.append(row)
    
    # Crear el Geodataframe completo
    gdf_completo = gpd.GeoDataFrame(data_list, crs="EPSG:4326")

    # Limpiar la columna sensor que viene como lista
    gdf_completo['sensor'] = gdf_completo['instruments'].apply(lambda x: x[0] if isinstance(x, list) and x else "N/A")

else:
    print(f"Error en la consulta: {response.status_code}")

print(f"\nDataFrame creado, número de imágenes obtenidas: {gdf_completo.shape[0]}")

# Fin script
print("Consulta de imágenes finalizada: ".upper() + strftime("%Y-%m-%d %H:%M:%S"))

Reducir y manipular Geodataframe intermedio

In [ ]:
# Columnas a mantener
columnas_finales = ['id', 'datetime', 'sensor', 'eo:cloud_cover', 'grid:code', 'platform', 'geometry']
print("Listado de columnas a mantener establecido")

# Nuevo GeoDataFrame filtrado
gdf_final = gdf_completo[columnas_finales].copy()

# Dimensionalidad del df
print(f'\nDataframe modificado. Número de filas: {gdf_final.shape[0]} | Número de columnas: {gdf_final.shape[1]}')
# print(gdf_final.to_string())

Filtrar imágenes para cada footprint

In [ ]:
# Calcular el área proyectando a EPSG:3857 (metros) y convertir a km2
gdf_final['area_km2'] = gdf_final.to_crs(epsg=3857).geometry.area / 1_000_000
print("Columna de área calculada")

# Agrupar imágenes por footprint y priorizar las que cubren mayor superficie con menor cantidad de nubes
gdf_ordenado = gdf_final.sort_values(
    by=['grid:code', 'area_km2', 'eo:cloud_cover'], 
    ascending=[True, False, True])
print("Imágenes agrupadas y ordenadas por footprint de forma correcta")

# Filtrar la mejor imagen para cada footprint
gdf_mejor_cobertura = gdf_ordenado.drop_duplicates(subset=['grid:code'], keep='first').copy()
print("Imágenes filtradas por footprint")

# Dimensionalidad del df y visualizar resultados
print(f'\nDataframe resultado. Número de filas: {gdf_mejor_cobertura.shape[0]} | Número de columnas: {gdf_mejor_cobertura.shape[1]}')
columnas_ver = ['id', 'grid:code', 'datetime', 'eo:cloud_cover', 'area_km2']
print(f"\n{gdf_mejor_cobertura[columnas_ver].to_string(index=False)}")

Exportar a Gejoson

In [ ]:
# Exportar a Geojson
gdf_mejor_cobertura.to_file("img_noclouds_sentinel.geojson", driver='GeoJSON')
print(f"Número de footprints exportados: {len(gdf_mejor_cobertura)}")